# Week 3 — Unsupervised Learning and Clustering Analysis

## Hands-on project: Wine segmentation with K-Means and Hierarchical Clustering

### Objective
Discover natural groups among wine samples using physicochemical measurements **without using the original wine class labels for model training**.

This notebook implements:
- data acquisition,
- preprocessing and scaling,
- K-Means clustering,
- elbow and silhouette analysis,
- PCA visualization,
- cluster profiling,
- hierarchical clustering,
- and interpretation of the resulting segments.


## 1. Public dataset

**UCI Machine Learning Repository — Wine dataset**

Official page:
https://archive.ics.uci.edu/dataset/109/wine

Official raw data file:
https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data

The original dataset contains 178 observations, 13 continuous chemical features, and a class label. The class label is retained only as a reference column and is excluded from the unsupervised feature matrix.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from week3_clustering import (
    FEATURE_NAMES,
    UCI_RAW_URL,
    cluster_profile,
    cluster_sizes,
    compare_clusterings,
    create_figures,
    download_wine_data,
    evaluate_k_values,
    fit_hierarchical,
    fit_kmeans,
    load_wine_csv,
    pca_projection,
    prepare_features,
    scale_features,
    select_k,
)


## 2. Acquire the dataset

The following cell downloads the current UCI raw file when internet access is available. The bundled sample is used automatically when it is not.


In [ ]:
RAW_DIR = PROJECT_ROOT / "data" / "raw"
raw_path = RAW_DIR / "wine.data"

try:
    if not raw_path.exists():
        download_wine_data(raw_path)
    df = load_wine_csv(raw_path)
    acquisition_mode = "official UCI dataset"
except Exception as exc:
    print("Official download unavailable:", exc)
    df = load_wine_csv(PROJECT_ROOT / "data" / "sample" / "wine_sample.csv")
    acquisition_mode = "bundled sample fixture"

print("Acquisition mode:", acquisition_mode)
print("Shape:", df.shape)
df.head()


## 3. Initial inspection

We first separate the numerical chemistry features from the original class label. The class label is not an input to clustering.


In [ ]:
print(df.info())
print("\nMissing values:")
print(df.isna().sum().sum())
print("\nClass counts (reference only):")
print(df["class"].value_counts().sort_index())


### Why exclude the class label?

Using the class label to train the clusters would turn the problem into supervised learning. Instead, the clustering algorithm receives only the 13 numerical chemical measurements and discovers groups from those features.


In [ ]:
X, y_reference = prepare_features(df)
X.head()


## 4. Feature scaling

In [ ]:
X_scaled, scaler = scale_features(X)

scaled_summary = pd.DataFrame(
    {
        "mean_after_scaling": X_scaled.mean(axis=0),
        "std_after_scaling": X_scaled.std(axis=0),
    },
    index=X.columns,
)
scaled_summary.head(13)


### Interpretation

Standardization gives each feature mean approximately 0 and standard deviation approximately 1. This prevents variables with larger numerical ranges from dominating Euclidean distance calculations.


## 5. Choosing the number of K-Means clusters

In [ ]:
metrics = evaluate_k_values(
    X_scaled,
    k_values=list(range(2, 8)),
    random_state=42,
)
metrics


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

axes[0].plot(metrics["k"], metrics["inertia"], marker="o")
axes[0].set_title("Elbow Curve")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

axes[1].plot(metrics["k"], metrics["silhouette_score"], marker="o")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette score")

fig.tight_layout()
plt.show()

selected_k = select_k(metrics)
print("Selected k based on highest silhouette score:", selected_k)


### Rationale

The Elbow Method looks for diminishing returns in within-cluster compactness. Silhouette analysis measures how well each point fits inside its assigned cluster relative to neighboring clusters. Using both avoids relying on a single heuristic.

The implementation selects the `k` with the highest silhouette score, breaking ties in favor of the smaller `k`.


## 6. Fit the final K-Means model

In [ ]:
kmeans, kmeans_labels = fit_kmeans(
    X_scaled,
    n_clusters=selected_k,
    random_state=42,
)

print("Inertia:", round(kmeans.inertia_, 3))
print("Cluster labels:", np.unique(kmeans_labels))
print("Cluster counts:")
print(cluster_sizes(kmeans_labels))


## 7. Visualize clusters with PCA

In [ ]:
projection, pca = pca_projection(X_scaled)

fig, ax = plt.subplots(figsize=(8.5, 5.5))
scatter = ax.scatter(
    projection[:, 0],
    projection[:, 1],
    c=kmeans_labels,
    s=50,
    alpha=0.85,
)
ax.set_title(
    "K-Means Clusters in PCA Space "
    f"({pca.explained_variance_ratio_[0]*100:.1f}% + "
    f"{pca.explained_variance_ratio_[1]*100:.1f}% variance)"
)
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
fig.colorbar(scatter, ax=ax, label="Cluster")
fig.tight_layout()
plt.show()


### Interpretation

PCA compresses the standardized 13-dimensional feature space into two dimensions for visualization. The plot helps show whether the clusters are visibly separated, but it does not contain all information present in the original feature space.


## 8. Profile each cluster

In [ ]:
profile = cluster_profile(X_scaled, kmeans_labels, FEATURE_NAMES)
profile


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(profile, cmap="vlag", center=0, annot=False, ax=ax)
ax.set_title("Cluster Profile — Mean Standardized Features")
ax.set_xlabel("Feature")
ax.set_ylabel("Cluster")
fig.tight_layout()
plt.show()


### How to interpret the profile

- Positive standardized values mean the cluster is above the overall mean for that feature.
- Negative values mean the cluster is below the overall mean.
- A cluster with consistently high values across a group of related features can be interpreted as having a stronger chemical signature along those dimensions.
- These are descriptive segments, not automatically business categories.


## 9. Feature-level boxplot summary

In [ ]:
profile_long = (
    profile.reset_index()
    .melt(id_vars="cluster", var_name="feature", value_name="mean_z")
)

fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(
    data=profile_long,
    x="cluster",
    y="mean_z",
    ax=ax,
)
ax.set_title("Distribution of Standardized Cluster-Profile Means")
ax.set_xlabel("Cluster")
ax.set_ylabel("Mean standardized feature value")
fig.tight_layout()
plt.show()


## 10. Hierarchical clustering

In [ ]:
hierarchical_model, hierarchical_labels = fit_hierarchical(
    X_scaled,
    n_clusters=selected_k,
)

comparison = compare_clusterings(kmeans_labels, hierarchical_labels)
comparison


### Interpretation

Agglomerative hierarchical clustering provides an independent unsupervised partition. The **Adjusted Rand Index (ARI)** compares the two cluster assignments while correcting for chance agreement. Higher agreement suggests that the broad segmentation is not unique to a single clustering algorithm.


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

linkage_matrix = linkage(X_scaled, method="ward")

fig, ax = plt.subplots(figsize=(12, 6))
dendrogram(linkage_matrix, no_labels=True, ax=ax)
ax.set_title("Hierarchical Clustering Dendrogram")
ax.set_xlabel("Sample index")
ax.set_ylabel("Ward distance")
fig.tight_layout()
plt.show()


## 11. Optional post-hoc comparison with the original class labels

In [ ]:
posthoc = pd.crosstab(
    pd.Series(y_reference, name="original_class"),
    pd.Series(kmeans_labels, name="kmeans_cluster"),
)
posthoc


### Important caution

The table above is **post-hoc descriptive comparison only**. The original class labels were not used to select features, scale data, choose assignments, or train the clustering model. This preserves the unsupervised nature of the main analysis.


## 12. Export all project outputs

In [ ]:
PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "outputs" / "figures"
TABLES = PROJECT_ROOT / "outputs" / "tables"

PROCESSED.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

clustered = X.copy()
clustered["cluster_kmeans"] = kmeans_labels
clustered["cluster_hierarchical"] = hierarchical_labels
clustered["class_reference_only"] = y_reference.astype("Int64")
clustered.to_csv(PROCESSED / "wine_clustered.csv", index=False)

X.to_csv(PROCESSED / "wine_clustering_ready.csv", index=False)
metrics.to_csv(TABLES / "cluster_metrics.csv", index=False)
profile.to_csv(TABLES / "cluster_profile.csv")
cluster_sizes(kmeans_labels).to_csv(TABLES / "cluster_sizes.csv", index=False)
comparison.to_csv(TABLES / "hierarchical_comparison.csv", index=False)

create_figures(
    X_scaled,
    metrics,
    kmeans_labels,
    profile,
    hierarchical_labels,
    FIGURES,
)

print("Exports complete.")


## 13. Key findings framework

A complete interpretation should answer:

1. **How many clusters were selected and why?**
   - Refer to the elbow curve and silhouette scores.

2. **How large is each cluster?**
   - Use the cluster-size table.

3. **What makes each cluster different?**
   - Use the standardized centroid/profile table and heatmap.

4. **Are the clusters visually separable?**
   - Inspect the PCA projection.

5. **Is the segmentation stable across algorithms?**
   - Use the hierarchical clustering comparison and ARI.

6. **What could the segmentation mean in practice?**
   - Discuss potential uses such as exploratory product segmentation, quality profiling, or identifying chemically similar sample groups.


## 14. Limitations and responsible interpretation

- K-Means assumes approximately spherical clusters in the chosen feature space.
- Results depend on scaling and the selected features.
- PCA is a visualization aid, not the clustering space.
- A high silhouette score does not prove that the clusters have real-world meaning.
- Hierarchical clustering can change with linkage choice and distance definition.
- The sample is relatively small, so generalization to a broader wine population should be cautious.
- Domain validation is needed before turning clusters into operational categories.


## 15. Conclusion

This notebook completes the Week 3 unsupervised-learning requirement through an end-to-end implementation of K-Means and hierarchical clustering. The project combines quantitative model-selection metrics, visual diagnostics, feature-level cluster profiling, and practical interpretation.

The repository is designed for GitHub submission and can be extended with additional clustering algorithms, dimensionality-reduction methods, stability analysis, or domain-specific validation.
